In [ ]:
from lsst.daf.butler import Butler
import pandas as pd
import numpy as np

import importlib
import smp_task_io

In [ ]:
importlib.reload(smp_task_io)

In [ ]:
instrument = "LSSTCam"
repo = "dp2_prep"
collection_stage4 = "LSSTCam/runs/DRP/DP2"
butler_stage4    = Butler(repo, collections=collection_stage4)

In [ ]:
file_path = "/home/g/garazigp/rubin-user/Stage/rubin_snia_lc/data/2704/monster/lightcurve_tract_2704_g_739161397740437575.csv"

df = pd.read_csv(file_path)
visits_to_load = set(zip(df["visitId"].astype(int), df["detectorId"].astype(int)))

In [ ]:
#visit_images = {}
visit_image_refs = []
for visit, det in list(visits_to_load)[:20]:
    try:
        visitImage = butler_stage4.getDeferred("visit_image", visit=visit, detector=det, instrument="LSSTCam")
        #visit_images[(visit, det)] = visitImage
        visit_image_refs.append(visitImage)
        print(f"OK  visit={visit}  det={det}")
    except Exception as e:
        print(f"ERR visit={visit}  det={det}  → {e}")

print(f"\n{len(visit_image_refs)} images loaded.")

In [ ]:
tract = 2704
diaObjectId = 739161397740437575
diaObject = butler_stage4.get("dia_object", tract=tract, skymap="lsst_cells_v2")
print(type(diaObject))

In [ ]:
#diaObject.columns

In [ ]:
ra, dec = [diaObject.loc[diaObjectId]['ra']], [diaObject.loc[diaObjectId]['dec']]
#print(ra)

In [ ]:
config = smp_task_io.SMPTaskConfig(ra=ra, dec=dec)
task = smp_task_io.SMPTask(config=config)

In [ ]:
data = task.run(visit_image_refs)

## **Scene Model**

In [ ]:
import sceneModel

In [ ]:
importlib.reload(sceneModel)

In [ ]:
data_arrays = []
weights = []
masks = []
psfs = []
wcss = []

for image_cutout in data.sceneModelingInputs['cutouts'] :
    data_arrays.append(image_cutout.data)
    weights.append(image_cutout.uncertainty.array**(-1))
    masks.append(image_cutout.flags)
    psfs.append(image_cutout.psf)
    wcss.append(image_cutout.wcs)

In [ ]:
photo_ratio = np.ones(len(data_arrays))
model = sceneModel.SceneModel(data.sceneModelingInputs['positions'][0], data_arrays, weights, masks, psfs,
                      wcss, photo_ratio, np.array(data.sceneModelingInputs['on_off']), imageSize=51, modelSize=55)

In [ ]:
#init_params = np.zeros(55**2 + len(data_arrays) + np.array(data.sceneModelingInputs['on_off']).sum() + 2)
init_galaxy = np.zeros((55,55))
init_galaxy[25:28, 15:18] = 1
init_params = np.concatenate([init_galaxy.ravel(), np.zeros(len(data_arrays) + np.array(data.sceneModelingInputs['on_off']).sum() + 2)])
models, resids = model.generateAbChi2(init_params)

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2, ax3) = plt.subplots(3, 3, figsize=(8,8))
ax1[0].imshow(data_arrays[0])
ax1[1].imshow(models[0])
ax1[2].imshow(resids[0])

ax2[0].imshow(data_arrays[1])
ax2[1].imshow(models[1])
ax2[2].imshow(resids[1])

ax3[0].imshow(data_arrays[2])
ax3[1].imshow(models[2])
ax3[2].imshow(resids[2])

In [ ]:
wcs = wcss[0]
wcs.pixel_to_world(0,0)

In [ ]:
import lsst.geom as geom

visitImage_test = butler_stage4.get('preliminary_visit_summary', visit=2025072300532, detector=154)

sky_coord = geom.SpherePoint(ra[0], dec[0], geom.degrees)
visitImage_wcs = visitImage_test.find(57).wcs  # detector 86 is outside
pixel_point = visitImage_wcs.skyToPixel(sky_coord)

print(pixel_point)

In [ ]:
len(model.xImage)

In [ ]:
REPO = "dp2_prep"
COLLECTION_STAGE2  = "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2"
butler_stage2 = Butler(REPO, collections=COLLECTION_STAGE2)

In [ ]:
visits_test = df["visitId"].astype(int)[:20]
det_test = df["detectorId"].astype(int)[:20]
print(len(visits_test))

vsummaries = []

for v_id in visits_test:
    v_summary  = butler_stage2.get('visit_summary', visit=v_id).asAstropy()
    detector = v_summary[v_summary['visit'] == v_id]
    vsummaries.append(v_summary)

In [ ]:
summ_df = vsummaries[
  ['id','bbox_min_x','bbox_min_y','bbox_max_x','bbox_max_y','visit',
     'physical_filter','band','psfSigma','psfArea','ra','dec',
     'zenithDistance','pixelScale','expTime','zeroPoint']
].to_pandas()

In [ ]:
ra_corner_0 = vsummaries['raCorners'].value[:,0]
ra_corner_1 = vsummaries['raCorners'].value[:,1]
ra_corner_2 = vsummaries['raCorners'].value[:,2]
ra_corner_3 = vsummaries['raCorners'].value[:,3]

dec_corner_0 = vsummaries['decCorners'].value[:,0]
dec_corner_1 = vsummaries['decCorners'].value[:,1]
dec_corner_2 = vsummaries['decCorners'].value[:,2]
dec_corner_3 = vsummaries['decCorners'].value[:,3]

summ_df['ccd_corner_1_ra'] = ra_corner_0
summ_df['ccd_corner_2_ra'] = ra_corner_1
summ_df['ccd_corner_3_ra'] = ra_corner_2
summ_df['ccd_corner_4_ra'] = ra_corner_3

summ_df['ccd_corner_1_dec'] = dec_corner_0
summ_df['ccd_corner_2_dec'] = dec_corner_1
summ_df['ccd_corner_3_dec'] = dec_corner_2
summ_df['ccd_corner_4_dec'] = dec_corner_3

In [ ]:
colors = plt.cm.tab10(np.linspace(0, 1, len(subset_df)+1))

fig, ax = plt.subplots(1, 1, figsize=(8,8))
# plt.grid()
for irow, arow in subset_df.iterrows():
    verts = (
        [arow['ccd_corner_1_ra'], arow['ccd_corner_1_dec']],
        [arow['ccd_corner_2_ra'], arow['ccd_corner_2_dec']],
        [arow['ccd_corner_3_ra'], arow['ccd_corner_3_dec']],
        [arow['ccd_corner_4_ra'], arow['ccd_corner_4_dec']],
        [arow['ccd_corner_1_ra'], arow['ccd_corner_1_dec']]   # repeat to close loop
    )

    rough_center = np.mean(np.array(verts), axis=0)

    xs, ys = zip(*verts)
    ax.plot(xs, ys, '--', color=colors[irow], alpha=0.9)
    ax.text(arow['ccd_corner_1_ra'], arow['ccd_corner_1_dec'], f"{arow['visit']}-{arow['id']}", alpha=1)

In [ ]:
test_tab = np.array([1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 4, 4, 4, 4, 5, 5, 5, 5, 6, 6, 6, 6])
print(int(len(test_tab)/4))
reshaped_tab = test_tab.reshape((int(len(test_tab)/4),4))
print(reshaped_tab)